# Barnacle instance segmentation!! Use colab + T4 

This is my notebook for the DALI challenge. The notebook fine-tunes **Mask R-CNN R50-FPN + PointRend**. Upload `barnacle_colab_final.zip` when prompted. 

Helpful links:

[hf detectron2/PointRend readme](https://huggingface.co/spaces/brjathu/HMR2.0/blob/6481b0b2587d1fce454473438770a7643bf3b134/vendor/detectron2/projects/PointRend/README.md)

[arxiv about PointRend](https://arxiv.org/html/1912.08193v2)


The notebook looks through the entire learning process. The first run is an exploration and single-tile training test. The later fine-tuning cell continues into tuning for the barnacles. 


In [ ]:
import sys, subprocess
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
assert torch.cuda.is_available(), "Select a T4 GPU runtime, then reconnect."
print("GPU:", torch.cuda.get_device_name())
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
subprocess.run(["nvidia-smi"], check=True)


## Upload the training bundle

The ZIP includes prepared image/label pairs, review artifacts, validation code, and the training scripts. 

The combined bundle already includes checkpoint resume, live loss plots, both unseen images and reporting code. No patch uploads are needed.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, os
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
assert len(archives) == 1, "Upload exactly one barnacle training ZIP."
EXTRACT_ROOT = Path("/content/barnacle_workspace")
assert not EXTRACT_ROOT.exists() or not any(EXTRACT_ROOT.iterdir()), "Choose a new EXTRACT_ROOT to preserve existing files."
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        destination = (EXTRACT_ROOT / member.filename).resolve()
        assert destination.is_relative_to(EXTRACT_ROOT.resolve()), "Unsafe archive path"
        assert (member.external_attr >> 16) & 0o170000 != 0o120000, "Symlinks are not supported"
    archive.extractall(EXTRACT_ROOT)
WORKSPACE = EXTRACT_ROOT / "barnacle_training"
DATASET = WORKSPACE / "dataset"
assert (DATASET / "dataset_summary.json").is_file()
print("Workspace:", WORKSPACE)
summary = json.loads((DATASET / "dataset_summary.json").read_text())
print("Reviewed training data:", summary["training_ready"])
for scene in summary["scenes"]:
    print(scene["id"], scene["split"], "tiles:", scene["tile_count"], "label count:", scene["candidate_instance_count"])
del uploaded


## Validate the uploaded data

Structural validation checks mask pixels, COCO geometry, tile coordinates, and split separation. It cannot establish biological correctness. The existing annotations need review; 55 ambiguous components were flagged in the initial build.


In [ ]:
subprocess.run([sys.executable, str(WORKSPACE / "training/train_pointrend.py"),
                "--dataset", str(DATASET), "--inspect-data"], check=True)
from IPython.display import display, Image
# This is a visualization of SUPPLIED LABELS, not model predictions.
display(Image(filename=str(DATASET / "scenes/img2/annotation_preview.jpg"), width=700))


## Install Detectron2 and PointRend

This compiles Detectron2 for the runtime GPU against Colab's existing PyTorch. Compilation can take several minutes. The setup checks CUDA and torchvision operators before building. Colab images change; if this cell fails, retain the error log rather than installing random CUDA/PyTorch versions.

The source commit and environment are recorded with each run. To reproduce a previous environment, replace `main` with its saved Detectron2 commit hash. The local Windows preparation environment is unaffected.


In [ ]:
command = [
    sys.executable,
    str(WORKSPACE / "training/colab_setup.py"),
    "--checkout", "/content/detectron2",
    "--revision", "main",
]

with open("/content/detectron2_install.log", "w") as log:
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
    return_code = process.wait()

print("\nInstaller exit code:", return_code)
assert return_code == 0, "Detectron2 installation failed; inspect the log before continuing."


## Choose persistent checkpoint storage

Google Drive is recommended so a disconnected runtime does not erase checkpoints. This cell asks you to connect Drive. Set `SAVE_TO_DRIVE=False` to keep everything in `/content`, then download checkpoints before ending the runtime.


In [ ]:
from datetime import datetime, timezone
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_ROOT = Path("/content/drive/MyDrive/BarnacleChallenge/runs")
else:
    RUNS_ROOT = Path("/content/barnacle_runs")
RUN_ROOT = RUNS_ROOT / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_ROOT.mkdir(parents=True, exist_ok=False)
print("Run outputs:", RUN_ROOT)


## First diagnostic: 50 steps on one tile

`ALLOW_PROVISIONAL=True` explicitly permits exploratory training with the current labels. Unresolved annotation areas are **not automatically ignored** by the generic trainer, so their missing/ambiguous labels can affect learning and metrics. Review/correct the dataset before treating results as scientific evidence. Set this to False for reviewed-data enforcement.

Smoke mode repeats one training tile, disables augmentation, and skips quantitative validation. It checks model loading, masks, gradients and checkpoint writing. Fifty steps are not a promise of convergence or good predictions. Warnings about COCO classification/mask output layers being reinitialized are expected when switching from 80 classes to one.


In [ ]:
ALLOW_PROVISIONAL = True  # Explicit diagnostic opt-in; current data is not reviewed.
DETECTRON2_ROOT = Path("/content/detectron2")
SMOKE_STEPS = 50
SMOKE_OUTPUT = RUN_ROOT / "smoke"
common = [sys.executable, str(WORKSPACE / "training/train_pointrend.py"),
          "--dataset", str(DATASET), "--detectron2-root", str(DETECTRON2_ROOT)]
if ALLOW_PROVISIONAL:
    common += ["--allow-provisional"]
sys.path.insert(0, str(WORKSPACE / "training"))
from live_colab import run_with_live_plot
run_with_live_plot(common + ["--mode", "smoke", "--steps", str(SMOKE_STEPS),
                         "--output", str(SMOKE_OUTPUT)], SMOKE_OUTPUT)

LATEST_OUTPUT = SMOKE_OUTPUT


## Inspect actual predictions and loss

Left: supplied labels. Right: actual predictions from the trained diagnostic model, displayed at score ? 0.5. A blank right panel can mean no prediction passed that threshold; it does not necessarily mean the code failed. Check loss, then extend the diagnostic or review labels/settings. A falling loss on the repeated tile demonstrates fitting, not generalization.


In [ ]:
import matplotlib.pyplot as plt
metrics = [json.loads(line) for line in (SMOKE_OUTPUT / "metrics.json").read_text().splitlines() if line.strip()]
loss_rows = [row for row in metrics if "total_loss" in row]
if loss_rows:
    plt.plot([row["iteration"] for row in loss_rows], [row["total_loss"] for row in loss_rows])
    plt.xlabel("Training iteration")
    plt.ylabel("Total training loss")
    plt.grid(alpha=0.3)
    plt.show()
for split in ("train", "val"):
    print(split, "? supplied labels left, model predictions right")
    display(Image(filename=str(SMOKE_OUTPUT / "previews" / f"{split}_comparison.jpg"), width=1000))
print("Checkpoint:", SMOKE_OUTPUT / "model_final.pth")


In [ ]:
import json
import cv2
from pycocotools import mask as mask_utils
from detectron2.utils.visualizer import Visualizer
from PIL import Image as PILImage

path = SMOKE_OUTPUT / "previews/val_predictions.json"
data = json.loads(path.read_text())
rgb = cv2.cvtColor(cv2.imread(data["image_file"]), cv2.COLOR_BGR2RGB)

masks = []
for instance in data["instances"]:
    rle = instance["segmentation"]
    h, w = rle["size"]
    masks.append(mask_utils.decode(mask_utils.frPyObjects(rle, h, w)))

view = Visualizer(rgb).overlay_instances(masks=masks, alpha=0.35)
display(PILImage.fromarray(view.get_image()))

## Optional: fine-tune on all nine training tiles

Run this only after inspecting the diagnostic. This is a **new run from pretrained weights**, not a continuation of the single-tile diagnostic. It uses flips and modest brightness/contrast changes, evaluates the 16 held-out validation tiles every 100 steps, and saves the best validation mask-AP checkpoint plus the final checkpoint.

600 steps is an initial experiment setting, not an established optimum. Training uses one source photograph, validation another; metrics are exploratory while labels remain provisional. The detection cap is 600, above the observed maximum 396 labels/tile; reported AP uses that custom cap rather than standard COCO's 100. Overlapping tile counts must not be summed into a photograph count.


In [ ]:
RUN_FINETUNE = False  # Change to True when ready to start this separate experiment.
FINETUNE_STEPS = 200
FINETUNE_OUTPUT = RUN_ROOT / "finetune"
if RUN_FINETUNE:
    run_with_live_plot(common + ["--mode", "finetune", "--steps", str(FINETUNE_STEPS),
                             "--output", str(FINETUNE_OUTPUT)], FINETUNE_OUTPUT)
    LATEST_OUTPUT = FINETUNE_OUTPUT
    print((FINETUNE_OUTPUT / "validation_metrics.json").read_text())
    display(Image(filename=str(FINETUNE_OUTPUT / "previews/val_comparison.jpg"), width=1000))
else:
    print("Skipped. Set RUN_FINETUNE=True after inspecting the diagnostic.")


## Optional: continue your latest checkpoint

Set `RUN_RESUME=True` and choose `SOURCE_RUN`: use `LATEST_OUTPUT` for the run just completed, or an explicit Drive path for an earlier session. The final checkpoint and its config/manifest must be together. This restores training state, adds `ADDITIONAL_STEPS`, and writes a new run directory. No patch upload is required. The live plot shows absolute step count, losses and ETA.


In [ ]:
RUN_RESUME = False
if RUN_RESUME:
    import sys, json, torch
    from pathlib import Path

    sys.path.insert(0, str(WORKSPACE / "training"))
    from live_colab import run_with_live_plot

    # Choose the run to continue.
    SOURCE_RUN = LATEST_OUTPUT  # Or Path("/content/drive/MyDrive/.../your_actual_run")
    CHECKPOINT = SOURCE_RUN / "model_final.pth"
    ADDITIONAL_STEPS = 200

    state = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
    completed = int(state["iteration"]) + 1
    del state

    target = completed + ADDITIONAL_STEPS
    previous = json.loads((SOURCE_RUN / "run_manifest.json").read_text())
    NEW_OUTPUT = SOURCE_RUN.parent / f"{SOURCE_RUN.name}_to_{target}"

    command = [
        sys.executable, str(WORKSPACE / "training/train_pointrend.py"),
        "--dataset", str(DATASET),
        "--detectron2-root", str(DETECTRON2_ROOT),
        "--mode", previous["mode"],
        "--tile-index", str(previous.get("training_tile_index") or 0),
        "--resume-from", str(CHECKPOINT),
        "--steps", str(target),
        "--output", str(NEW_OUTPUT),
        "--allow-provisional",
    ]

    print(f"Continuing from {completed} to {target} total steps")
    run_with_live_plot(command, NEW_OUTPUT)

    # Subsequent prediction cells should use this newly trained model.
    RUN_OUTPUT = LATEST_OUTPUT = NEW_OUTPUT
else:
    print("Resume skipped. Set RUN_RESUME=True to continue the selected checkpoint.")


## Optional: report on the two unseen images and preserve the model

Set `RUN_UNSEEN_DEMO=True`. The inputs and reporting code arrived in the first ZIP; no further upload is needed. `LATEST_OUTPUT` follows each successful training/resume run. To evaluate an existing model without retraining, set `TRAINED_RUN` to its actual Drive folder. The demo first archives the selected final checkpoint, then crops and predicts on both unseen images. Reports compare thresholds 0.2, 0.3 and 0.5. These are unlabeled images, so reports cannot measure accuracy.


In [ ]:
RUN_UNSEEN_DEMO = False
if RUN_UNSEEN_DEMO:
    sys.path.insert(0, str(WORKSPACE / "training"))
    from demo_colab import run_unseen_demo
    assert (WORKSPACE / "demo/demo_inputs.json").is_file(), "Upload the combined final bundle first."
    TRAINED_RUN = LATEST_OUTPUT  # Or Path("/content/drive/MyDrive/.../your_actual_run")
    print("Using trained run:", TRAINED_RUN)
    DEMO_ROOT = run_unseen_demo(
        WORKSPACE, TRAINED_RUN,
        checkpoint_name="model_final.pth",
        score_threshold=0.3,
    )
else:
    print("Unseen demo skipped. Set RUN_UNSEEN_DEMO=True when ready.")


## Keep these artifacts

- `model_final.pth`: final model and training state; `model_best.pth` in fine-tune runs: best validation mask AP.
- `config.yaml`, `run_manifest.json`, `environment.json`: settings, label status, source commit, and runtime versions.
- `metrics.json`: losses and periodic validation metrics; `validation_metrics.json`: final fine-tune metrics.
- `previews/*_comparison.jpg`: clearly separated labels and predictions; `*_predictions.json`: lossless predicted masks and scores in tile coordinates.

If using Drive, outputs are already in the printed run folder. Otherwise download checkpoints from Colab's Files panel before disconnecting. The optional unseen demo provides tiled inference and duplicate reconciliation for whole-photo predictions and counts. Ruler calibration and scientific morphometrics are separate subsequent work.

References: [PointRend](https://github.com/facebookresearch/detectron2/tree/main/projects/PointRend), [Detectron2 installation](https://detectron2.readthedocs.io/en/latest/tutorials/install.html).
